# LAB: Chaos Agent: Tools, Lore, Questionable Judgment

## Objective

Build a **creative complaint handler** for *NormalObjects* with a LangChain agent.
The agent picks its own tools, so you must **prove** the answer comes from tool output.

```
complaint → agent (LLM) → picks tools (any order) → tool outputs → final answer
```

## Steps

| # | Step | Code |
|---|------|---|
| 1 | Setup | PROVIDED |
| 2 | Create creative tools | PROVIDED + YOUR TURN |
| 3 | Create the agent | YOUR TURN |
| 4 | Test with sample complaints | YOUR TURN |
| 5 | Analyze agent behavior | YOUR TURN |
| | `lab_proof.md` | YOUR TURN |

**Submit:** `lab_proof.md` with code, one complaint, tool evidence, final output, one failure.

| Label | Meaning |
|---|---|
| **PROVIDED** | Run and read it |
| **YOUR TURN** | Replace the `# TODO` lines |
| **OPTIONAL** | Not required |

**Reference:** `langchain_integration.ipynb` (tools + `create_agent`)

**Prompt discipline:** use AI to review or debug, not to write your answer.

---
## Step 1: Setup [PROVIDED]

Add `OPENAI_API_KEY=...` to a `.env` file next to this notebook.

In [ ]:
# Install required packages (uncomment to run)
# !pip install -U langchain langchain-openai python-dotenv

In [ ]:
import os
import random
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

load_dotenv()

# Make sure OPENAI_API_KEY is set in your .env file
print("OpenAI API Key loaded:", "OPENAI_API_KEY" in os.environ)

model = init_chat_model(model="gpt-4o-mini", model_provider="openai", temperature=0.7)

**Done when:** the key prints `True` and there are no import errors.

---
## Step 2: Create Creative Tools

Every tool needs:
1. the `@tool` decorator
2. a clear docstring (the agent reads it to choose the tool)

### 2a. `consult_demogorgon` [PROVIDED]

In [ ]:
@tool
def consult_demogorgon(complaint: str) -> str:
    """Consult the Demogorgon about a complaint involving the Upside Down.
    Returns a chaotic, unreliable opinion."""
    responses = [
        f"The Demogorgon sniffs '{complaint}' and roars. It seems... hungry, not helpful.",
        f"The Demogorgon tilts its petal-face at '{complaint}'. Interpreted as: 'turn off the lights'.",
        f"The Demogorgon ignores '{complaint}' and walks back into the wall. Classic.",
    ]
    return random.choice(responses)

### 2b. `check_hawkins_records` [YOUR TURN]

- Dictionary with 4 keys: `portal`, `monsters`, `psychics`, `electricity`
- Return the record whose key appears in the query (case-insensitive)
- No match → `"No record found ..."`

In [ ]:
# YOUR TURN: Hawkins records tool

# TODO: @tool
# TODO: def check_hawkins_records(query: str) -> str:
#           """..."""
#           records = {
#               "portal": "...",
#               "monsters": "...",
#               "psychics": "...",
#               "electricity": "...",
#           }
#           ...

### 2c. `cast_interdimensional_spell` [YOUR TURN]

- Args: `problem`, `creativity_level="medium"`
- Map `low/medium/high` → `1/2/3`
- 4 spell strings that include the problem
- `random.sample()` N spells, join with `"\n"`

In [ ]:
# YOUR TURN: spell tool

# TODO: @tool
# TODO: def cast_interdimensional_spell(problem: str, creativity_level: str = "medium") -> str:
#           """..."""
#           ...

### 2d. `gather_party_wisdom` [YOUR TURN]

- Dictionary with 4 keys, one per party member (Mike, Dustin, Lucas, Will)
- Return the match (case-insensitive)
- No match → generic "the party huddles..." answer

In [ ]:
# YOUR TURN: party wisdom tool

# TODO: @tool
# TODO: def gather_party_wisdom(question: str) -> str:
#           """..."""
#           ...

In [ ]:
# YOUR TURN: test each tool on its own with .invoke({...})

# TODO: print(consult_demogorgon.invoke({"complaint": "My lights keep flickering"}))
# TODO: print(check_hawkins_records.invoke({"query": "tell me about the portal"}))
# TODO: print(cast_interdimensional_spell.invoke({"problem": "Demodogs in my garden", "creativity_level": "high"}))
# TODO: print(gather_party_wisdom.invoke({"question": "what is the plan?"}))

**Done when:** all 4 tools return text, and a query with no matching key returns your fallback.

---
## Step 3: Create the Agent [YOUR TURN]

System prompt must:
- encourage creative tool use
- force at least one tool call
- say: answer **only** from tool results
- ask the agent to list the tools it used

**Hint:** `create_agent(model=..., tools=..., system_prompt=...)` in `langchain_integration.ipynb`.

In [ ]:
# YOUR TURN: tool list + system prompt + agent

# TODO: tools = [...]

# TODO: system_prompt = """..."""

# TODO: agent = create_agent(model=model, tools=tools, system_prompt=system_prompt)

# TODO: print([t.name for t in tools])

**Done when:** the agent is created and prints the 4 tool names.

---
## Step 4: Test with Sample Complaints [YOUR TURN]

Use `config={"recursion_limit": 10}` to stop loops.

In [ ]:
complaints = [
    "My Christmas lights keep flickering and I hear noises from the wall. What is going on?",
    "There is a weird portal in my basement. What's the plan?",
    "A Demodog ate my homework. I need a creative solution, creativity level high!",
    "My neighbor's cat is acting weird. Is it from the Upside Down?",  # no record exists
]

# YOUR TURN: run every complaint, keep the results for Step 5

# TODO: results = []
# TODO: for complaint in complaints:
#           result = agent.invoke({"messages": [HumanMessage(content=complaint)]}, config={"recursion_limit": 10})
#           results.append(result)
#           print(...)   # complaint + final answer

**Check:** for the cat complaint, does the agent admit "no record", or invent one?

---
## Step 5: Analyze Agent Behavior [YOUR TURN]

### 5a. Track tool usage

For each result: print the tool order (`a → b → c`). Then count total uses per tool.

**Hint:** AI messages have `.tool_calls`; use `collections.Counter`.

In [ ]:
# YOUR TURN: tool order + counts

# TODO: from collections import Counter
# TODO: for r in results:
#           order = [...]
#           print(" → ".join(order))
# TODO: print counts

### 5b. Trace one answer (grounding)

For one result, print every tool call, its args, and what it returned.

**Hint:** messages with `msg.type == "tool"` hold the tool output.

In [ ]:
# YOUR TURN: evidence trace for one result

# TODO: for msg in results[0]["messages"]:
#           ...   # print tool calls (ai) and tool outputs (tool)
# TODO: print(results[0]["messages"][-1].content)

**Check:** mark each claim in the final answer as either from a tool or invented.

### 5c. Compare with a structured approach

Write `structured_handler(complaint)`: always calls records → party → spell (`low`), no LLM.

In [ ]:
# YOUR TURN: fixed-order handler

# TODO: def structured_handler(complaint: str) -> str:
#           ...

# TODO: print(structured_handler(complaints[1]))

| | Agent | Structured |
|---|---|---|
| Same tools every run? | | |
| Tool order predictable? | | |
| Can invent facts? | | |
| Easy to trace? | | |
| Best for which client case? | | |

---
## Extensions [OPTIONAL]

- More tools (`consult_eleven`, `check_government_files`)
- LLM call inside a tool
- Memory with `InMemorySaver` + `thread_id`
- Streamlit / Gradio UI
- Measure response time and tool calls per complaint

In [ ]:
# YOUR TURN (optional)

# TODO

---
## Write `lab_proof.md` [YOUR TURN]

```markdown
# Lab Proof: NormalObjects Creative Complaint Handler

## 1. What I built
## 2. Query
## 3. Tool evidence (calls + args + outputs)
## 4. Final output
## 5. Trace (which claim comes from which tool output?)
## 6. Failure or limitation (where can it give a plausible but unsupported answer?)
## 7. Agent vs. structured handler
```

**Before you submit:**
- [ ] No API keys or `.env`
- [ ] Only lab files
- [ ] `lab_proof.md` is the entry point

---
### Troubleshooting

| Problem | Fix |
|---|---|
| Agent doesn't use tools | Clearer docstrings; prompt must require tool use |
| Tools in "wrong" order | Expected for agents. Need fixed order? Use LangGraph |
| Agent loops | Lower `recursion_limit`, improve docstrings |
| Generic answers | More specific tool outputs, lower temperature |
| `ImportError` on `create_agent` | `pip install -U langchain` (needs v1+) |